In [ ]:
!pip install -U bitsandbytes transformers

In [ ]:
import os
import math
import random
import glob
from google.colab import drive
import json
from PIL import Image
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
from google.colab import drive
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
drive.mount("/content/drive")

# Download model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-3.3B")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-3.3B", torch_dtype=torch.float16)

In [ ]:
model = model.to(device)

# Translate

In [ ]:
def translate_batch(texts, batch_size=32):
  src_lang = "eng_Latn"
  tgt_lang = "kaz_Cyrl"

  translations = []

  for i in tqdm(range(0, len(texts), batch_size), desc="Translating:"):
    batch = texts[i: i + batch_size]

    inputs = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    ).to(device)

    with torch.no_grad():
      outs = model.generate(
          **inputs,
          forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
          max_length=128
      )

    decoded = tokenizer.batch_decode(outs, skip_special_tokens=True)
    translations.extend(decoded)

  return translations

In [ ]:
english_captions = [
    "A dog running on green grass.",
    "A child playing with a red ball.",
    "A man in a suit standing near a building.",
    "A large plate of food on a wooden table."
]

In [ ]:
kazakh_captions = translate_batch(english_captions, batch_size=16)

In [ ]:
kazakh_captions

In [ ]:
for kk, en in zip(kazakh_captions, english_captions):
  print(f"KK: {kk}")
  print(f"EN: {en}")
  print(20*"--")

# Download COCO2014

In [ ]:
import os

# 1. Создаем структуру
root_dir = "/content/coco_karpathy"
os.makedirs(f"{root_dir}/images", exist_ok=True)
os.makedirs(f"{root_dir}/annotations", exist_ok=True)

# 2. Скачиваем Картинки 2014 (Train + Val) ~13GB + 6GB
# Важно: В Karpathy split картинки из train2014 и val2014 смешиваются
!wget -c http://images.cocodataset.org/zips/train2014.zip -O {root_dir}/train2014.zip
!unzip -q {root_dir}/train2014.zip -d {root_dir}/images/
!rm {root_dir}/train2014.zip

!wget -c http://images.cocodataset.org/zips/val2014.zip -O {root_dir}/val2014.zip
!unzip -q {root_dir}/val2014.zip -d {root_dir}/images/
!rm {root_dir}/val2014.zip

# 3. Скачиваем SAM Karpathy Split JSON (Это "Святой Грааль" для капшнинга)
# В этом JSON уже прописаны train/test/val сплиты, которые используют ВСЕ ученые.
!wget -c https://cs.stanford.edu/people/karpathy/deepimagesent/caption_datasets.zip -O {root_dir}/annotations/caption_datasets.zip
!unzip -q {root_dir}/annotations/caption_datasets.zip -d {root_dir}/annotations/
!rm {root_dir}/annotations/caption_datasets.zip

# Файл, который тебе нужен будет в коде: dataset_coco.json
print(f"✅ Готово! Твой главный файл аннотаций: {root_dir}/annotations/dataset_coco.json")

In [ ]:
os.listdir("coco_karpathy")

In [ ]:
images_path = os.path.join("coco_karpathy", "images")
annotations_path = os.path.join("coco_karpathy", "annotations")
images_path, annotations_path

In [ ]:
os.listdir(annotations_path)

In [ ]:
karpathy_json_path = "/content/coco_karpathy/annotations/dataset_coco.json"

In [ ]:
with open(karpathy_json_path, 'r') as f:
  data = json.load(f)

len(data)

In [ ]:
data

In [ ]:
data["images"]

In [ ]:
os.listdir(images_path)

In [ ]:
train_images_path = os.path.join(images_path, "train2014")
val_images_path = os.path.join(images_path, "val2014")
len(os.listdir(train_images_path)), len(os.listdir(val_images_path))

In [ ]:
os.listdir(train_images_path)[:5]

In [ ]:
image_path = os.path.join(train_images_path, 'COCO_train2014_000000121676.jpg')
Image.open(image_path).convert("RGB")

In [ ]:
train_images = [img for img in data["images"] if img["split"] in ["train", "restval"]]
print(f"len of train images: {len(train_images)}")

# Translate from English to Kazakh language

In [ ]:
kazakh_dataset = []
batch_size = 32
checkpoint_interval = 2500

In [ ]:
for i, img in tqdm(enumerate(train_images), total=len(train_images), desc="Translating")

In [ ]:
import json
import os
from tqdm.auto import tqdm

# --- 1. ПУТИ ---
karpathy_json_path = "/content/coco_karpathy/annotations/dataset_coco.json"
output_dir = "/content/drive/MyDrive/Vision KazLM/COCO_Data" # Сохраняем сразу на Диск!
os.makedirs(output_dir, exist_ok=True)
final_save_path = os.path.join(output_dir, "coco_kazakh_train_FULL.json")

# --- 2. ЗАГРУЗКА JSON ---
print("📂 Loading Karpathy JSON...")
with open(karpathy_json_path, 'r') as f:
    data = json.load(f)
print(f"✅ Loaded! Total images in JSON: {len(data['images'])}")

# --- 3. ФИЛЬТРАЦИЯ (Берем train + restval) ---
# Karpathy split использует 'train' и 'restval' для обучения
train_images = [img for img in data['images'] if img['split'] in ['train', 'restval']]
print(f"🔥 Images selected for training (train + restval): {len(train_images)}")

# --- 4. ПАЙПЛАЙН ПЕРЕВОДА ---
# Мы будем создавать новый список словарей
kazakh_dataset = []
batch_size_sentences = 128 # NLLB ест батчи быстро
checkpoint_interval = 2500 # Сохранять каждые 500 картинок

# Буферы для накопления
current_batch_texts = []
current_batch_indices = [] # (image_idx, sentence_idx) чтобы знать, куда вернуть перевод

print("🚀 Starting Translation Pipeline...")

# Проходим по всем картинкам
for i, img in tqdm(enumerate(train_images), total=len(train_images), desc="Processing Images"):

    image_filename = img['filename']
    image_filepath = img['filepath'] # 'train2014' или 'val2014'

    # Создаем структуру для новой записи
    new_entry = {
        "image_path": os.path.join(image_filepath, image_filename), # Полный путь
        "split": img['split'],
        "captions_en": [],
        "captions_kk": [] # Сюда положим перевод
    }

    # Собираем все 5 капшнов одной картинки
    for sent in img['sentences']:
        raw_text = sent['raw']
        new_entry["captions_en"].append(raw_text)

        # Добавляем в очередь на перевод
        current_batch_texts.append(raw_text)
        # Запоминаем: "Этот текст принадлежит картинке i"
        current_batch_indices.append(i)

    kazakh_dataset.append(new_entry)

    # --- КОГДА НАБРАЛСЯ БАТЧ ИЛИ КОНЕЦ ---
    if len(current_batch_texts) >= batch_size_sentences or i == len(train_images) - 1:
        if len(current_batch_texts) > 0:
            # 1. Переводим пачку!
            translated_texts = translate_batch(current_batch_texts, batch_size=batch_size_sentences)

            # 2. Раскладываем переводы обратно по картинкам
            for txt_kk, img_idx in zip(translated_texts, current_batch_indices):
                # Находим нужную картинку в нашем списке и добавляем перевод
                # Хитрость: img_idx соответствует индексу в kazakh_dataset, так как мы идем по порядку
                kazakh_dataset[img_idx]["captions_kk"].append(txt_kk)

            # 3. Очищаем буферы
            current_batch_texts = []
            current_batch_indices = []

    # --- СОХРАНЕНИЕ ЧЕКПОИНТА ---
    if (i + 1) % checkpoint_interval == 0:
        ckpt_path = os.path.join(output_dir, f"checkpoint_{i+1}.json")
        # Чтобы не забивать диск, можно перезаписывать один файл, но для безопасности лучше так
        # Или просто сохраняй текущий прогресс
        with open(ckpt_path, 'w', encoding='utf-8') as f:
            json.dump(kazakh_dataset[:i+1], f, ensure_ascii=False, indent=2)

# --- ФИНАЛЬНОЕ СОХРАНЕНИЕ ---
print("💾 Saving FULL dataset...")
with open(final_save_path, 'w', encoding='utf-8') as f:
    json.dump(kazakh_dataset, f, ensure_ascii=False, indent=2)

print(f"🎉 DONE! Saved to {final_save_path}")

In [ ]:
import json
import os
import glob
from tqdm.auto import tqdm

# --- 1. ПУТИ ---
karpathy_json_path = "/content/coco_karpathy/annotations/dataset_coco.json"
output_dir = "/content/drive/MyDrive/Vision KazLM/COCO_Data"
final_save_path = os.path.join(output_dir, "coco_kazakh_train_FULL.json")

# --- 2. ПОИСК ПОСЛЕДНЕГО ЧЕКПОИНТА ---
print("🔍 Searching for checkpoints...")
list_of_files = glob.glob(os.path.join(output_dir, 'checkpoint_*.json'))

if list_of_files:
    latest_file = max(list_of_files, key=os.path.getctime)
    print(f"✅ Found latest checkpoint: {latest_file}")

    with open(latest_file, 'r') as f:
        kazakh_dataset = json.load(f)

    start_index = len(kazakh_dataset)
    print(f"🔄 Resuming from image index: {start_index}")
else:
    print("⚠️ No checkpoints found. Starting from ZERO.")
    kazakh_dataset = []
    start_index = 0

# --- 3. ЗАГРУЗКА ИСХОДНИКА ---
print("📂 Loading Karpathy JSON...")
with open(karpathy_json_path, 'r') as f:
    data = json.load(f)

# Фильтруем (train + restval)
train_images = [img for img in data['images'] if img['split'] in ['train', 'restval']]
print(f"🔥 Total images to process: {len(train_images)}")

# Отрезаем то, что уже сделали
remaining_images = train_images[start_index:]
print(f"🚀 Images left to translate: {len(remaining_images)}")

# --- 4. НАСТРОЙКИ ---
batch_size_sentences = 128
checkpoint_interval = 2500

current_batch_texts = []
current_batch_indices = []

print("▶️ Continuing Translation Pipeline...")

# ВАЖНО: enumerate начинаем с start_index, чтобы логика чекпоинтов не сломалась
for i, img in tqdm(enumerate(remaining_images, start=start_index), total=len(remaining_images), desc="Processing"):

    image_filename = img['filename']
    image_filepath = img['filepath']

    new_entry = {
        "image_path": os.path.join(image_filepath, image_filename),
        "split": img['split'],
        "captions_en": [],
        "captions_kk": []
    }

    for sent in img['sentences']:
        raw_text = sent['raw']
        new_entry["captions_en"].append(raw_text)
        current_batch_texts.append(raw_text)
        current_batch_indices.append(i) # 'i' здесь уже с учетом сдвига (напр. 50000, 50001...)

    # Добавляем ПУСТУЮ запись в общий список (заполним её после перевода)
    kazakh_dataset.append(new_entry)

    # --- ПЕРЕВОД БАТЧА ---
    if len(current_batch_texts) >= batch_size_sentences or i == len(train_images) - 1:
        if len(current_batch_texts) > 0:
            translated_texts = translate_batch(current_batch_texts, batch_size=batch_size_sentences)

            for txt_kk, img_idx in zip(translated_texts, current_batch_indices):
                # img_idx указывает на правильное место в большом списке
                kazakh_dataset[img_idx]["captions_kk"].append(txt_kk)

            current_batch_texts = []
            current_batch_indices = []

    # --- СОХРАНЕНИЕ ---
    # Сохраняем, если прошли интервал ИЛИ если это самый последний файл
    if (i + 1) % checkpoint_interval == 0:
        ckpt_path = os.path.join(output_dir, f"checkpoint_{i+1}.json")
        with open(ckpt_path, 'w', encoding='utf-8') as f:
            json.dump(kazakh_dataset, f, ensure_ascii=False, indent=2)
            print(f"\n💾 Checkpoint saved: {i+1} images")

# --- ФИНАЛ ---
print("💾 Saving FULL dataset...")
with open(final_save_path, 'w', encoding='utf-8') as f:
    json.dump(kazakh_dataset, f, ensure_ascii=False, indent=2)

print(f"🎉 DONE! Saved to {final_save_path}")